In [1]:
%reload_ext autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import IPython.display as ipd
import whisper
import sys
sys.path.append("/home/romolo/VT1/coqui-tts")
from model_conf import ModelPaths, load_tts_and_trainer

/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [21]:
paths = ModelPaths()
tts, model, train_model, config = load_tts_and_trainer(paths)

 > Using model: xtts
>> DVAE weights restored from: /home/romolo/VT1/coqui-tts/XTTS_v2.0_original_model_files/dvae.pth


In [4]:
import torchaudio

def crop_audio(input_path, output_path, duration_sec=5):
    waveform, sample_rate = torchaudio.load(input_path)
    num_samples = int(duration_sec * sample_rate)
    cropped_waveform = waveform[:, :num_samples]
    torchaudio.save(output_path, cropped_waveform, sample_rate)

# Example usage:
crop_audio("/home/romolo/VT1/coqui-tts/data/target_no_sr.wav", "/home/romolo/VT1/coqui-tts/data/TARGET_5s.wav", duration_sec=5)


In [5]:
import torch
import torchaudio
import noisereduce as nr
import numpy as np
import librosa
import soundfile as sf

def denoise_audio(input_path, output_path):
    # Load audio
    waveform, sample_rate = torchaudio.load(input_path)

    # Convert to mono if stereo
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    audio_np = waveform.squeeze(0).numpy().astype(np.float32)

    # --- Trim silence using librosa ---
    trimmed, _ = librosa.effects.trim(audio_np, top_db=50)
    audio_np = trimmed.astype(np.float32)

    # --- Apply light denoising ---
    reduced_noise = nr.reduce_noise(
        y=audio_np,
        sr=sample_rate,
        stationary=False,
        prop_decrease=0.7,
        n_fft=1024
    )

    # Normalize
    reduced_noise = reduced_noise / np.max(np.abs(reduced_noise) + 1e-6)

    # --- Save with torchaudio or soundfile ---
    sf.write(output_path, reduced_noise, sample_rate)
    print(f"Denoised audio saved to: {output_path}")

denoise_audio(
    "/home/romolo/VT1/coqui-tts/data/target.wav",
    "/home/romolo/VT1/coqui-tts/data/TARGET_no_noise.wav"
)

Denoised audio saved to: /home/romolo/VT1/coqui-tts/data/TARGET_no_noise.wav


In [6]:
asr_model = whisper.load_model("base")

In [7]:
import os

In [8]:
ref_samples = os.listdir("/home/romolo/VT1/coqui-tts/test_data/Dataset/references/26/")
ref_samples = ["/home/romolo/VT1/coqui-tts/test_data/Dataset/references/26/" + i for i in ref_samples]

In [37]:
orig_target_sample = "/home/romolo/VT1/coqui-tts/data/target.wav"
text = asr_model.transcribe(orig_target_sample)['text']

In [38]:
target_sample = orig_target_sample
for i in range(0,10):
    wav = model.forward_from_audios_and_text('en',text,target_sample,ref_samples[0],train_model,config.model_args.max_conditioning_length, config.model_args.min_conditioning_length)
    tts.synthesizer.save_wav(wav=wav['wav'], path=f"/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_{i}.wav")
    target_sample = f"/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_{i}.wav"

In [39]:
ipd.Audio(wav['wav'], rate=24000)

In [40]:
from huggingface_hub import hf_hub_download

# automatically checks for cached file, optionally set `cache_dir` location
model_file = hf_hub_download(repo_id='Jenthe/ECAPA2', filename='ecapa2.pt', cache_dir=None)

In [41]:
import torch
import torchaudio
import torch.nn.functional as F

ecapa2 = torch.jit.load(model_file, map_location='cuda')


In [42]:
def calc_sim(aud1,aud2):
    #cossim between embeddings
    audio, sr = torchaudio.load(aud1)
    audio = torchaudio.functional.resample(audio, orig_freq=sr, new_freq=16_000)# sample rate of 16 kHz expected
    embedding = ecapa2(audio.to('cuda'))
    ref_audio, sr = torchaudio.load(aud2) # sample rate of 16 kHz expected
    ref_audio = torchaudio.functional.resample(ref_audio, orig_freq=sr, new_freq=16_000)
    ref_embedding = ecapa2(ref_audio.to('cuda'))
    sim = F.cosine_similarity(embedding, ref_embedding)
    return sim

In [44]:
for i in range(0,10):
    targ = calc_sim(f"/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_{i}.wav","/home/romolo/VT1/coqui-tts/data/target.wav")
    print(f"sample {i}")
    print(f"sim -- target to output: {targ}")
    ref = calc_sim(f"/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_{i}.wav",ref_samples[0])
    print(f"sim -- ref to output: {ref}")


sample 0
sim -- target to output: tensor([0.1321], device='cuda:0')
sim -- ref to output: tensor([0.5890], device='cuda:0')
sample 1
sim -- target to output: tensor([0.1854], device='cuda:0')
sim -- ref to output: tensor([0.6718], device='cuda:0')
sample 2
sim -- target to output: tensor([0.2040], device='cuda:0')
sim -- ref to output: tensor([0.6945], device='cuda:0')
sample 3
sim -- target to output: tensor([0.1917], device='cuda:0')
sim -- ref to output: tensor([0.6693], device='cuda:0')
sample 4
sim -- target to output: tensor([0.1758], device='cuda:0')
sim -- ref to output: tensor([0.7084], device='cuda:0')
sample 5
sim -- target to output: tensor([0.1740], device='cuda:0')
sim -- ref to output: tensor([0.7053], device='cuda:0')
sample 6
sim -- target to output: tensor([0.1812], device='cuda:0')
sim -- ref to output: tensor([0.7101], device='cuda:0')
sample 7
sim -- target to output: tensor([0.1960], device='cuda:0')
sim -- ref to output: tensor([0.7156], device='cuda:0')
sample 8